In [ ]:
import math

import equinox as eqx
import jax
import jax.numpy as jnp
from jaxtyping import Array, Int, PRNGKeyArray

## Naive implementation

A direct JAX/Equinox translation of https://gist.github.com/huchenxucs/c65524185e8e35c4bcfae4059f896c16

In [2]:
class RelativePositionBias(eqx.Module):
    bidirectional: bool = eqx.field(static=True)
    num_buckets: int = eqx.field(static=True)
    max_distance: int = eqx.field(static=True)
    num_heads: int = eqx.field(static=True)

    embedding: eqx.nn.Embedding

    def __init__(
        self,
        bidirectional: bool = True,
        num_buckets: int = 32,
        max_distance=128,
        num_heads=2,
        *,
        key: PRNGKeyArray,
    ):
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.num_heads = num_heads
        self.embedding = eqx.nn.Embedding(self.num_buckets, self.num_heads, key=key)

    def _relative_position_bucket(
        self,
        relative_position: Int[Array, "query_size key_size"],
    ):
        """
        Adapted from Mesh Tensorflow:
        https://github.com/tensorflow/mesh/blob/0cb87fe07da627bf0b7e60475d59f95ed6b5be3d/mesh_tensorflow/transformer/transformer_layers.py#L593
        Translate relative position to a bucket number for relative attention.
        The relative position is defined as memory_position - query_position, i.e.
        the distance in tokens from the attending position to the attended-to
        position.  If bidirectional=False, then positive relative positions are
        invalid.
        We use smaller buckets for small absolute relative_position and larger buckets
        for larger absolute relative_positions.  All relative positions >=max_distance
        map to the same bucket.  All relative positions <=-max_distance map to the
        same bucket.  This should allow for more graceful generalization to longer
        sequences than the model has been trained on.
        Args:
            relative_position: an int32 Tensor
            bidirectional: a boolean - whether the attention is bidirectional
            num_buckets: an integer
            max_distance: an integer
        Returns:
            a Tensor with the same shape as relative_position, containing int32
            values in the range [0, num_buckets)
        """
        ret = 0
        n = -relative_position
        if self.bidirectional:
            num_buckets = self.num_buckets // 2
            ret += (n < 0).astype(
                int
            ) * num_buckets  # mtf.to_int32(mtf.less(n, 0)) * num_buckets
            n = jnp.abs(n)
        else:
            num_buckets = self.num_buckets
            n = jnp.maximum(n, jnp.zeros_like(n))
        # now n is in the range [0, inf)

        # half of the buckets are for exact increments in positions
        max_exact = num_buckets // 2
        is_small = n < max_exact
        # The other half of the buckets are for logarithmically bigger bins in positions up to max_distance
        val_if_large = max_exact + (
            jnp.log(n.astype(float) / max_exact)
            / math.log(self.max_distance / max_exact)
            * (num_buckets - max_exact)
        ).astype(int)
        val_if_large = val_if_large.astype(int)
        val_if_large = jnp.minimum(
            val_if_large, jnp.full_like(val_if_large, num_buckets - 1)
        )

        ret += jnp.where(is_small, n, val_if_large)
        return ret

    def compute_bias(self, qlen, klen):
        """Compute binned relative position bias"""
        context_position = jnp.arange(qlen, dtype=int)[:, None]
        memory_position = jnp.arange(klen, dtype=int)[None, :]
        relative_position = memory_position - context_position  # shape (qlen, klen)
        """
                   k
             0   1   2   3
        q   -1   0   1   2
            -2  -1   0   1
            -3  -2  -1   0
        """
        rp_bucket = self._relative_position_bucket(
            relative_position,  # shape (qlen, klen)
        )
        print(rp_bucket)
        values = jax.vmap(jax.vmap(self.embedding))(
            rp_bucket
        )  # shape (qlen, klen, num_heads)
        values = jnp.expand_dims(jnp.transpose(values, axes=(2, 0, 1)), axis=0)
        return values

    def forward(self, qlen, klen):
        return self.compute_bias(qlen, klen)  # shape (1, num_heads, qlen, klen)

## Alternative implementation

See context_flux_no.nn.position_encoding.relative.T5RelativePositionalEncoding

In [3]:
from context_flux_no.nn.position_encoding import T5RelativePositionalEncoding

In [4]:
pos_enc = T5RelativePositionalEncoding(
    num_encodings=16,
    encoding_size=2,
    max_distance=16,
    bidirectional=True,
    key=jax.random.key(0),
)

In [5]:
pos_enc_naive = RelativePositionBias(
    bidirectional=True,
    num_buckets=16,
    max_distance=16,
    num_heads=2,
    key=jax.random.key(0),
)

In [6]:
query_size = 32
key_size = 32
relative_position = jnp.arange(key_size, dtype=int) - jnp.arange(
    query_size, dtype=int
).reshape(-1, 1)
relative_position

Array([[  0,   1,   2, ...,  29,  30,  31],
       [ -1,   0,   1, ...,  28,  29,  30],
       [ -2,  -1,   0, ...,  27,  28,  29],
       ...,
       [-29, -28, -27, ...,   0,   1,   2],
       [-30, -29, -28, ...,  -1,   0,   1],
       [-31, -30, -29, ...,  -2,  -1,   0]], dtype=int32)

In [7]:
bin1 = pos_enc.relative_position_to_embedding_bin(relative_position)

In [8]:
bin2 = pos_enc_naive._relative_position_bucket(relative_position)

In [9]:
bin2

Array([[ 0,  9, 10, ..., 15, 15, 15],
       [ 1,  0,  9, ..., 15, 15, 15],
       [ 2,  1,  0, ..., 15, 15, 15],
       ...,
       [ 7,  7,  7, ...,  0,  9, 10],
       [ 7,  7,  7, ...,  1,  0,  9],
       [ 7,  7,  7, ...,  2,  1,  0]], dtype=int32)

In [10]:
pos_enc.embedding.weight

Array([[ 1.6226422 ,  2.0252647 ],
       [-0.43359444, -0.07861735],
       [ 0.1760909 , -0.97208923],
       [-0.49529874,  0.4943786 ],
       [ 0.6643493 , -0.9501635 ],
       [ 2.1795304 , -1.9551506 ],
       [ 0.35857072,  0.15779513],
       [ 1.2770847 ,  1.5104648 ],
       [ 0.970656  ,  0.59960806],
       [ 0.0247007 , -1.9164772 ],
       [-1.8593491 ,  1.728144  ],
       [ 0.04719035,  0.814128  ],
       [ 0.13132767,  0.28284705],
       [ 1.2435943 ,  0.6902801 ],
       [-0.80073744, -0.74099   ],
       [-1.5388287 ,  0.30269185]], dtype=float32)

In [11]:
pos_enc_naive.embedding.weight

Array([[ 1.6226422 ,  2.0252647 ],
       [-0.43359444, -0.07861735],
       [ 0.1760909 , -0.97208923],
       [-0.49529874,  0.4943786 ],
       [ 0.6643493 , -0.9501635 ],
       [ 2.1795304 , -1.9551506 ],
       [ 0.35857072,  0.15779513],
       [ 1.2770847 ,  1.5104648 ],
       [ 0.970656  ,  0.59960806],
       [ 0.0247007 , -1.9164772 ],
       [-1.8593491 ,  1.728144  ],
       [ 0.04719035,  0.814128  ],
       [ 0.13132767,  0.28284705],
       [ 1.2435943 ,  0.6902801 ],
       [-0.80073744, -0.74099   ],
       [-1.5388287 ,  0.30269185]], dtype=float32)

In [12]:
enc_naive = pos_enc_naive.forward(32, 32)

[[ 0  9 10 ... 15 15 15]
 [ 1  0  9 ... 15 15 15]
 [ 2  1  0 ... 15 15 15]
 ...
 [ 7  7  7 ...  0  9 10]
 [ 7  7  7 ...  1  0  9]
 [ 7  7  7 ...  2  1  0]]


In [13]:
enc = pos_enc(32, 32)

[[ 0  9 10 ... 15 15 15]
 [ 1  0  9 ... 15 15 15]
 [ 2  1  0 ... 15 15 15]
 ...
 [ 7  7  7 ...  0  9 10]
 [ 7  7  7 ...  1  0  9]
 [ 7  7  7 ...  2  1  0]]


In [14]:
enc_naive

Array([[[[ 1.6226422 ,  0.0247007 , -1.8593491 , ..., -1.5388287 ,
          -1.5388287 , -1.5388287 ],
         [-0.43359444,  1.6226422 ,  0.0247007 , ..., -1.5388287 ,
          -1.5388287 , -1.5388287 ],
         [ 0.1760909 , -0.43359444,  1.6226422 , ..., -1.5388287 ,
          -1.5388287 , -1.5388287 ],
         ...,
         [ 1.2770847 ,  1.2770847 ,  1.2770847 , ...,  1.6226422 ,
           0.0247007 , -1.8593491 ],
         [ 1.2770847 ,  1.2770847 ,  1.2770847 , ..., -0.43359444,
           1.6226422 ,  0.0247007 ],
         [ 1.2770847 ,  1.2770847 ,  1.2770847 , ...,  0.1760909 ,
          -0.43359444,  1.6226422 ]],

        [[ 2.0252647 , -1.9164772 ,  1.728144  , ...,  0.30269185,
           0.30269185,  0.30269185],
         [-0.07861735,  2.0252647 , -1.9164772 , ...,  0.30269185,
           0.30269185,  0.30269185],
         [-0.97208923, -0.07861735,  2.0252647 , ...,  0.30269185,
           0.30269185,  0.30269185],
         ...,
         [ 1.5104648 ,  1.5104648 

In [15]:
enc

Array([[[ 1.6226422 ,  2.0252647 ],
        [ 0.0247007 , -1.9164772 ],
        [-1.8593491 ,  1.728144  ],
        ...,
        [-1.5388287 ,  0.30269185],
        [-1.5388287 ,  0.30269185],
        [-1.5388287 ,  0.30269185]],

       [[-0.43359444, -0.07861735],
        [ 1.6226422 ,  2.0252647 ],
        [ 0.0247007 , -1.9164772 ],
        ...,
        [-1.5388287 ,  0.30269185],
        [-1.5388287 ,  0.30269185],
        [-1.5388287 ,  0.30269185]],

       [[ 0.1760909 , -0.97208923],
        [-0.43359444, -0.07861735],
        [ 1.6226422 ,  2.0252647 ],
        ...,
        [-1.5388287 ,  0.30269185],
        [-1.5388287 ,  0.30269185],
        [-1.5388287 ,  0.30269185]],

       ...,

       [[ 1.2770847 ,  1.5104648 ],
        [ 1.2770847 ,  1.5104648 ],
        [ 1.2770847 ,  1.5104648 ],
        ...,
        [ 1.6226422 ,  2.0252647 ],
        [ 0.0247007 , -1.9164772 ],
        [-1.8593491 ,  1.728144  ]],

       [[ 1.2770847 ,  1.5104648 ],
        [ 1.2770847 ,  1.51

In [17]:
from einops import rearrange


jnp.array_equal(enc, rearrange(enc_naive, "1 c q k -> q k c"))

Array(True, dtype=bool)